In [13]:
from numpy import array
from keras.models import Sequential
from keras.layers import LSTM, Dense
# Split a univariate sequence into samples
def split_sequence(sequence, n_steps):
   X, y = list(), list()
   for i in range(len(sequence)):
       end_ix = i + n_steps
       if end_ix > len(sequence)-1:
           break
       seq_x, seq_y = sequence[i:end_ix], sequence[end_ix]
       X.append(seq_x)
       y.append(seq_y)
   return array(X), array(y)
# Define input sequence
raw_seq = [10, 20, 30, 40, 50, 60, 70, 80, 90]
n_steps = 3
X, y = split_sequence(raw_seq, n_steps)
# Reshape from [samples, timesteps] into [samples, timesteps, features]
n_features = 1



In [10]:
i = 0
end_ix = 3
raw_seq = [10, 20, 30, 40, 50, 60, 70, 80, 90]
seq_x, seq_y = raw_seq[i:end_ix], raw_seq[end_ix]
print(seq_x)
print(seq_y)

[10, 20, 30]
40


In [15]:
X.shape

(6, 3)

In [11]:
X = X.reshape((X.shape[0], X.shape[1], n_features))
X

array([[[10],
        [20],
        [30]],

       [[20],
        [30],
        [40]],

       [[30],
        [40],
        [50]],

       [[40],
        [50],
        [60]],

       [[50],
        [60],
        [70]],

       [[60],
        [70],
        [80]]])

In [16]:
model = Sequential()
model.add(LSTM(50, activation='relu', input_shape=(n_steps, n_features)))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse')
# Fit model
model.fit(X, y, epochs=200, verbose=0)
# Demonstrate prediction
x_input = array([70, 80, 90])
x_input = x_input.reshape((1, n_steps, n_features))
yhat = model.predict(x_input, verbose=0)
print(yhat)

C:\Users\ranji\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


[[103.58127]]


In [1]:
import pandas as pd
import glob
import os
import matplotlib.pyplot as plt
import numpy as np

In [2]:
import os
os.chdir(os.path.expanduser(r"H:\GitHub\AD_Behavioral_Modeling"))

In [3]:
from data.preprocess import preprocess_all
df = preprocess_all(r"H:\GitHub\AD_Behavioral_Modeling\data")

H:\GitHub\AD_Behavioral_Modeling\data\preprocess.py:156: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('Vehicle_Global_ID', group_keys=False).apply(label_lane_changes)


lane_change
0    1216811
1      36816
2       9051
Name: count, dtype: int64


In [4]:
from extract_features import calculate_features
df = calculate_features(df)

Calculating Individual Dynamics...
Calculating Lead Vehicle Features...
Converting to SI units....
Feature Engineering Done..!!


In [5]:
features = [
        "v_vel", "a_long", "v_lat", "Lane_ID", "v_lat_lag_5", "v_lat_lag_10", 
        "lat_displacement_1s", "can_go_right", "can_go_left", "a_long_std_1s", 
        "TTC", "actual_gap", "gap_rate_trend_1s", "rel_speed"
    ]

In [6]:
from utils.data_prep import data_split_with_sampling
X_train, X_test, y_train, y_test = data_split_with_sampling(df, features, sampling_keep_factor=4)

Original Training Size: 1012841
Under-sampled Training Size: 189415
New Class Distribution:
 lane_change
0    151532
1     30216
2      7667
Name: count, dtype: int64


In [40]:
from models.lstm import LaneChangeLSTM
from data.create_lstm_dataset import LaneChangeSequenceDataset
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, f1_score
import numpy as np

In [11]:
window_size = 30 # 3 seconds of history at 10Hz
train_dataset = LaneChangeSequenceDataset(X_train, y_train, df.loc[X_train.index, 'Vehicle_Global_ID'], window_size)
test_dataset = LaneChangeSequenceDataset(X_test, y_test, df.loc[X_test.index, 'Vehicle_Global_ID'], window_size)


In [41]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [43]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LaneChangeLSTM(input_size=len(features), hidden_size=64, num_layers=2).to(device)

In [45]:
weights = torch.tensor([1.0, 3.0, 10.0]).to(device)
criterion = torch.nn.CrossEntropyLoss(weight=weights)
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [46]:
print(f"Training LSTM on {device}...")
for epoch in range(10):
    model.train()
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1} Loss: {loss.item():.4f}")

Training LSTM on cpu...
Epoch 1 Loss: 0.5254


KeyboardInterrupt: 